# Skyra TSER Recon

### 1. Import Libraries

In [7]:
import mapvbvd
import numpy as np 
import matplotlib.pyplot as plt
import scipy.signal as sg
%matplotlib qt

### 2. Load dataset

In [8]:
def load_TSER(file, points, echoes, channels = 15):
    twixObj = mapvbvd.mapVBVD(file)
    twixObj[1].image.squeeze = True #remove extra dimensions
    twixObj[1].image.flagRemoveOS = False #no readout oversampling
    data = twixObj[1].image[''].reshape(points,channels,-1, 2, echoes) # arrange data into (points, channels, slices, phase cycle, echoes)
    data_pc = data[:,:,:,0] + data[:,:,:,1] # add phase cycle steps to remove ringing

    # data_pc is phase cycled data with shape (points, channels, slices, echoes)
    return data_pc

data = load_TSER('Data/CMIF/Oct5/meas_MID00101_FID02502_400a_std_200.dat', 128, 400)

pymapVBVD version 0.7.0
Software version: VD


### 3. Fourier Transform to get Image

In [18]:
data_ft = np.fft.fftshift(np.fft.fft(np.fft.fftshift(data[3:-3,:,:,:], axes=(0)), axis=0), axes=(0))

## Sensitivity adjustment: add all coil signals together, scaled by the 5th echo amplitude for each coil
data_sa = np.einsum('jkhi,jkh->jhi', data_ft, data_ft[:, :, :, 5])

### 4. Plot 3D data for one slice

In [19]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.view_init(elev=30, azim=-26, roll=0)

distance = np.linspace(-54,54,np.shape(data_ft)[0])
colors = plt.cm.viridis(np.linspace(0, 1, 400))

for i in range(400):
    ax.plot(i, distance, np.abs(data_sa[:,0,i]),color=colors[i], lw=1) #every point, slice 0, echo 'i'

plt.show()

### 5. Plot Slice Cross Section

In [26]:
plt.figure()
plt.plot(np.abs(data_sa[:,0,5])) #Every point, channel 6, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 5.5 Plot Plate Map

In [ ]:
fig, ax = plt.subplots(1,1)
ax.imshow(np.log(np.abs(data_sa[:,:,50])), aspect='auto') #Every point, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 6. Plot Decay for One Point

In [34]:
point = 21
echoes = 400
te = 10e-3
plt.figure()
time = np.arange(echoes)*te
plt.plot(time, np.abs(data_sa[53,0,:])) #Point 52, slice 0, every echo
plt.xlabel("Distance (mm)")
plt.ylabel("Amplitude")
plt.show()

In [35]:
import scipy.optimize as opt
n_wells = 12
well_pitch = 9e-3
te = 10e-3


def get_t2(echo_data, te):
    def _decay(x, a, b, c):
        return a * np.exp(-b * x)+c
    popt, _ = opt.curve_fit(_decay, np.arange(len(echo_data))*te, echo_data/np.max(echo_data))
    return 1/popt[1]

n_slices = np.shape(data_sa)[1]
points = np.shape(data_sa)[0]
slice_peaks = {}
slice_t2s = {}
## full plate processing
for slice_idx in range(n_slices):
    model = np.concatenate((np.zeros(2), np.ones(5), np.zeros(2)))
    target = np.convolve(np.abs(data_sa[:,slice_idx,5]), model)
    peaks, _ = sg.find_peaks(target, prominence=0.05*np.std(data_sa[:,0,5]))
    slice_peaks[slice_idx] = peaks - 4
    slice_t2s[slice_idx] = []
    for peak in slice_peaks[slice_idx]:
        slice_t2s[slice_idx].append(get_t2(np.abs(data_sa[peak,slice_idx,:]), te))


print(slice_peaks)
print(slice_t2s)

# for slice_idx in range(np.shape(data_sa)[1]):

    # peaks, properties = sg.find_peaks(data_ft[30, 6, ], prominence=1, width=20)
# plt.plot(model)


{0: array([14, 23, 53, 63]), 1: array([14, 23, 43, 63]), 2: array([14, 23, 73, 82])}
{0: [np.float64(1.528972835259295), np.float64(2.3179288567990612), np.float64(2.5770500148821687), np.float64(2.210055807778917)], 1: [np.float64(1.8599355057648796), np.float64(0.3369917494758769), np.float64(0.569604296455307), np.float64(1.9835541582127523)], 2: [np.float64(1.9920643541557452), np.float64(0.9669513253931813), np.float64(0.30462844044743365), np.float64(1.5587213803857367)]}


In [64]:
def find_first_row(slice_peaks_dict):
    #estimate first row position
    first_row_idx = np.inf
    for i in range(n_slices):
        for peak in slice_peaks[i]:
            if(peak < first_row_idx):
                first_row_idx = peak
    return first_row_idx

first_row = find_first_row(slice_peaks)
points = np.shape(data_sa)[0]
points_per_well = points/n_wells

#assign to wells
well_t2s = np.full((n_slices, n_wells), np.nan)

for i in range(n_slices):
    for peak_idx, peak in enumerate(slice_peaks[i]):
        points = peak - first_row
        well = int((points + points_per_well/2)/points_per_well)
        if(np.isnan(well_t2s[i, well])):
            well_t2s[i, well] = slice_t2s[i][peak_idx]
        else:
            print('two points fall in same well')

In [65]:
fig, ax = plt.subplots(1,1)
ax.imshow(well_t2s, aspect='auto') #Every point, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

In [55]:
print(well_t2s[0,0] == np.nan)

False


In [ ]:
plt.figure()
print(np.shape(data_sa))
# plt.plot(np.abs(data_sa[:,:,5]))


# plt.plot(model/1000000)
plt.plot(np.convolve(np.abs(data_sa[:,1,5]), model))
plt.plot(np.convolve(np.abs(data_sa[:,2,5]), model))
# plt.plot(np.abs(data_ft[:,0,1,5])) #Every point, channel 6, slice 0, echo 5
# plt.plot(pat/1000) #Every point, channel 6, slice 0, echo 5
# plt.plot(np.convolve(pat, np.abs(data_ft[:,0,0,5]), mode='valid'))
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

(122, 3, 400)
